In [1]:
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat


probe_data = loadmat("/media/ubuntu/sda/duan/rat/probe/chanMapQPX_mice1.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y * 3/5

probe = Probe()
probe.set_contacts(positions=probe_position, contact_ids=probe_data['chanMap'][:, 0])


probe.set_device_channel_indices(range(128))

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
all_rhd_files = os.listdir("/media/ubuntu/sda/mouse_test/raw_data/JUAN/20260115-juan1_260115_161743")
all_rhd_files.remove('settings.xml')
all_rhd_files  = sorted(all_rhd_files)

recording_raw_list = []
for i, rhd_file in enumerate(all_rhd_files):
    recording_raw_list.append(se.read_intan(f'/media/ubuntu/sda/mouse_test/raw_data/JUAN/20260115-juan1_260115_161743/{rhd_file}', stream_id='0'))

all_rhd_files = os.listdir("/media/ubuntu/sda/mouse_test/raw_data/JUAN/20260305_juan/20260305_juan1_260305_103912")
all_rhd_files.remove('settings.xml')
all_rhd_files  = sorted(all_rhd_files)

recording_raw_list_2 = []
for i, rhd_file in enumerate(all_rhd_files):
    recording = se.read_intan(f'/media/ubuntu/sda/mouse_test/raw_data/JUAN/20260305_juan/20260305_juan1_260305_103912/{rhd_file}', stream_id='0')
    recording_raw_list_2.append(recording)
   

In [3]:
recording_raw = concatenate_recordings(recording_list=recording_raw_list)
recording_raw_2 = concatenate_recordings(recording_list=recording_raw_list_2)

In [4]:
#recording_raw_2 = recording_raw_2.select_channels(recording_raw_2.channel_ids[128:])
recording_raw_2 = recording_raw_2.rename_channels(recording_raw.channel_ids)

In [5]:
recording_raw = concatenate_recordings([recording_raw, recording_raw_2])
recording_raw = spre.unsigned_to_signed(recording_raw)
recording_raw = spre.resample(recording_raw, 10000)
recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)

In [6]:
output_folder = '/media/ubuntu/sda/mouse_test/sorted/JUAN/juan1_260115_260305'
recording_preprocessed = recording_f.save(format="binary", n_jobs = 30)

Use cache_folder=/tmp/spikeinterface_cache/tmp81hpz3cd/I913F8UO
write_binary_recording 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=2.44 MiB - total_memory=73.24 MiB - chunk_duration=1.00s


write_binary_recording (workers: 30 processes): 100%|██████████| 5668/5668 [01:36<00:00, 58.60it/s]


In [ ]:
output_folder = '/media/ubuntu/sda/mouse_test/sorted/JUAN/juan1_260115_260305'
recording_preprocessed = recording_f.save(format="binary", n_jobs = 30)

default_params = {
        'detect_sign': -1,  
        'adjacency_radius': 120, 
        'freq_min': 300,  
        'freq_max': 3000,
        'filter': True,
        'whiten': True,  
        'num_workers': 30,
        'clip_size': 50,
        'detect_threshold': 5,
        'detect_interval': 3,  
    }
sorting_mountainsort = ss.run_sorter(sorter_name='mountainsort4',
                                recording=recording_preprocessed,
                                remove_existing_folder='True',
                                folder=output_folder,
                                **default_params)

analyzer_mountainsort = si.create_sorting_analyzer(
    sorting=sorting_mountainsort, 
    recording=recording_preprocessed, 
    format='binary_folder', 
    folder=output_folder + '/analyzer_kilosort4_binary'
)

# 计算扩展信息
extensions_to_compute = [
    "random_spikes",
    "waveforms",
    "noise_levels",
    "templates",
    "unit_locations",
    "spike_locations",
    "correlograms",
    "template_similarity"
]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_mountainsort.compute(extensions_to_compute, extension_params=extension_params, n_jobs = 20)

# 读取spikes.npy并检查无效的spike
spikes_path = output_folder + "/analyzer_kilosort4_binary/sorting/spikes.npy"
spikes = np.load(spikes_path)

# 获取recording的总样本数
total_samples = recording_f.get_num_samples()

# 检查第一个和最后一个spike
first_spike_valid = spikes[0]['sample_index'] >= 0
last_spike_valid = spikes[-1]['sample_index'] < total_samples

# 如果第一个或最后一个spike无效，删除所有无效的spike
if not first_spike_valid or not last_spike_valid:
    # 创建有效spike的掩码：sample_index >= 0 且 < total_samples
    valid_mask = (spikes['sample_index'] >= 0) & (spikes['sample_index'] < total_samples)
    spikes_filtered = spikes[valid_mask]
    
    # 保存过滤后的spikes
    np.save(spikes_path, spikes_filtered)
    print(f"删除了 {len(spikes) - len(spikes_filtered)} 个无效的spike")
    print(f"原始spike数量: {len(spikes)}, 过滤后: {len(spikes_filtered)}")
else:
    print("所有spike都在有效范围内")

qm_params = sqm.get_default_qm_params()
analyzer_mountainsort.compute("quality_metrics", qm_params, n_jobs = 20)

# 导出到phy格式
import spikeinterface.exporters as sexp
sexp.export_to_phy(analyzer_mountainsort, output_folder + "/phy_folder_for_kilosort", verbose=True, n_jobs = 20)

Use cache_folder=/tmp/spikeinterface_cache/tmpad4ira7v/NPAFVDL5
write_binary_recording 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=2.44 MiB - total_memory=73.24 MiB - chunk_duration=1.00s


write_binary_recording (workers: 30 processes): 100%|██████████| 5668/5668 [01:56<00:00, 48.66it/s]


Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording


estimate_sparsity (no parallelization): 100%|██████████| 5668/5668 [00:00<00:00, 11626.71it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/basesorting.py:380: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
noise_level (workers: 20 processes): 100%|██████████| 20/20 [00:00<00:00, 154.94it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/core/sortinganalyzer.py:2701: DeprecationWarning: The method 'cosine_similarity' is deprecated and will be removed in the next version. Use 'cosine' instead.
  params = self._set_params(**params)
Compute : spike_locations (workers: 20 processes): 100%|██████████| 5668/5668 [00:01<00:00, 4571.13it/s]
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/quality_metric_calculator

所有spike都在有效范围内


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1505: DeprecationWarning: `row_stack` alias is deprecated. Use `np.vstack` directly.
  inds_confidence90 = np.row_stack(np.where(conf_matrix[:, test_rp_centers_mask] > 0.9))
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1059: UserWarning: Some units have too few spikes : amplitude_cutoff is set to NaN
  warnings.warn(f"Some units have too few spikes : amplitude_cutoff is set to NaN")
extract PCs (workers: 20 processes): 100%|██████████| 5668/5668 [01:06<00:00, 85.69it/s] 

Run:
phy template-gui  /media/ubuntu/sda/mouse_test/sorted/JUAN/juan1_260115_260305/phy_folder_for_kilosort/params.py


In [7]:
def process_unified_sorting(
    recording_cmr,
    output_folder,
    distance_threshold: float = 10.0,
    similarity_threshold: float = 0.95,
    peak_sign: str= 'neg',
    n_jobs: int = 20,
    verbose: bool = False
):
    from spikeinterface.qualitymetrics import compute_quality_metrics
    import scipy
    from scipy.sparse.csgraph import connected_components
    import pickle
    sampling_frequency = recording_cmr.get_sampling_frequency()

    if verbose:
        print(f"\n{'='*60}")
        print(f"开始统一处理整个recording的sorting结果")
        print(f"{'='*60}\n")

    # 统一的phy_folder路径
    phy_folder = f'{output_folder}/phy_folder_for_kilosort'

    # 读取整个recording的sorting结果
    if verbose:
        print("读取统一的sorting结果...")
    sorting_curated_phy = se.read_phy(phy_folder, exclude_cluster_groups=["noise", "mua"])
    if verbose:
        print(f"读取到 {len(sorting_curated_phy.unit_ids)} 个units\n")

    # 创建analyzer并计算extensions
    if verbose:
        print("创建analyzer并计算extensions...")
    analyzer_curated_phy = si.create_sorting_analyzer(
        sorting=sorting_curated_phy,
        recording=recording_cmr,
        format='binary_folder',
        folder=output_folder + '/analyzer_curated_temp',
        n_jobs=n_jobs,
        verbose=verbose
    )

    extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "templates",
        "unit_locations",
        "template_similarity"
    ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_curated_phy.compute(
        extensions_to_compute,
        extension_params=extension_params,
        n_jobs=n_jobs,
        verbose=verbose
    )
    if verbose:
        print("完成extensions计算\n")

    # 获取neuron信息
    templates_ext = analyzer_curated_phy.get_extension("templates")
    templates_dense = templates_ext.data["average"]
    sparsity = analyzer_curated_phy.sparsity
    unit_locations_ext = analyzer_curated_phy.get_extension("unit_locations")
    unit_locations = unit_locations_ext.get_data()
    channel_locations = analyzer_curated_phy.get_channel_locations()

    # 处理merge逻辑
    if unit_locations.shape[1] >= 2:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations[:, :2],
            unit_locations[:, :2],
            metric="euclidean"
        )
    else:
        unit_distances = scipy.spatial.distance.cdist(
            unit_locations,
            unit_locations,
            metric="euclidean"
        )

    template_similarity_ext = analyzer_curated_phy.get_extension("template_similarity")
    template_similarity = template_similarity_ext.get_data()

    num_units = len(analyzer_curated_phy.unit_ids)
    pair_mask = np.zeros((num_units, num_units), dtype=bool)

    for i in range(num_units):
        for j in range(i + 1, num_units):
            if unit_distances[i, j] < distance_threshold and template_similarity[i, j] > similarity_threshold:
                pair_mask[i, j] = True
                pair_mask[j, i] = True

    n_components, labels = connected_components(
        csgraph=pair_mask,
        directed=False,
        return_labels=True
    )

    merge_unit_groups = []
    unit_ids_list = analyzer_curated_phy.unit_ids
    for component_id in range(n_components):
        unit_indices = np.where(labels == component_id)[0]
        if len(unit_indices) > 1:
            group = [unit_ids_list[i] for i in unit_indices]
            merge_unit_groups.append(group)

    if len(merge_unit_groups) > 0:
        print(f"发现 {len(merge_unit_groups)} 组需要merge的units，开始merge...")
        analyzer_merged = analyzer_curated_phy.merge_units(
            merge_unit_groups=merge_unit_groups,
            censor_ms=0.3,
            merging_mode="hard",
            new_id_strategy="append",
            format='binary_folder',
            folder=output_folder + '/analyzer_merged',
            verbose=True,
            n_jobs=20
        )
        
        analyzer_merged.compute(extensions_to_compute, extension_params=extension_params, n_jobs=20, verbose = False)
        
        templates_ext_final = analyzer_merged.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_merged.sparsity
        unit_locations_ext_final = analyzer_merged.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_merged.get_channel_locations()
        sorting_final = analyzer_merged.sorting
        unit_ids_list_final = analyzer_merged.unit_ids
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_merged.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = si.get_template_extremum_channel(
            analyzer_merged, 
            peak_sign=peak_sign,
            outputs="id"
        )
        
        channel_ids_list = list(analyzer_merged.recording.get_channel_ids())
    else:
        print("无需merge units\n")
        # 不需要merge，使用原始结果
        templates_ext_final = analyzer_curated_phy.get_extension("templates")
        templates_dense_final = templates_ext_final.data["average"]
        sparsity_final = analyzer_curated_phy.sparsity
        unit_locations_ext_final = analyzer_curated_phy.get_extension("unit_locations")
        unit_locations_final = unit_locations_ext_final.get_data()
        channel_locations_final = analyzer_curated_phy.get_channel_locations()
        sorting_final = analyzer_curated_phy.sorting
        unit_ids_list_final = unit_ids_list
        
        # 生成position_waveforms
        position_waveforms_final = []
        for unit_id in unit_ids_list_final:
            unit_index = analyzer_curated_phy.sorting.id_to_index(unit_id)
            template_dense_unit = templates_dense_final[unit_index, :, :]
            template_sparse_unit = sparsity_final.sparsify_waveforms(template_dense_unit[np.newaxis, :, :], unit_id)[0]
            sparse_channel_indices = sparsity_final.unit_id_to_channel_indices[unit_id]
            
            if len(sparse_channel_indices) == 0:
                position_waveform = np.zeros(templates_dense_final.shape[1], dtype=templates_dense_final.dtype)
                position_waveforms_final.append(position_waveform)
                continue
            
            sparse_channel_locations = channel_locations_final[sparse_channel_indices, :2]
            unit_location = unit_locations_final[unit_index, :2]
            
            distances = np.sqrt(np.sum((sparse_channel_locations - unit_location[np.newaxis, :])**2, axis=1))
            epsilon = 1e-10
            weights = 1.0 / (distances + epsilon)
            weights = weights / np.sum(weights)
            
            position_waveform = np.dot(template_sparse_unit, weights)
            position_waveforms_final.append(position_waveform)
        
        position_waveforms_final = np.array(position_waveforms_final)
        extremum_channels_final = si.get_template_extremum_channel(
            analyzer_curated_phy, 
            peak_sign=peak_sign,
            outputs="id"
        )
        
        channel_ids_list = list(analyzer_curated_phy.recording.get_channel_ids())

    print("计算每个unit的channel_id...")
    channel_ids_dict = {}  # {unit_id: [contact_id1, contact_id2, ...]}
    for idx, unit_id in enumerate(unit_ids_list_final):
        unit_index = sorting_final.id_to_index(unit_id)
        template_unit = templates_dense_final[unit_index, :, :]  # (n_samples, n_channels)
        
        # 找到template中值不为0的通道
        non_zero_channels = []
        for ch_idx in range(template_unit.shape[1]):  # 遍历channels（最后一个维度）
            if np.any(template_unit[:, ch_idx] != 0):  # 检查该通道在所有时间点的值
                # recording的channel_id已经是contact_id，直接使用
                contact_id = str(channel_ids_list[ch_idx])
                non_zero_channels.append(contact_id)
        
        channel_ids_dict[unit_id] = non_zero_channels

    print(f"完成channel_id计算，共处理{len(channel_ids_dict)}个units\n")

    # 计算channel_snr（每个unit的各个channel的SNR）
    print("计算channel_snr...")
    duration_samples = int(10 * sampling_frequency)  # 10秒
    max_samples = min(duration_samples, recording_cmr.get_num_samples())
    traces = recording_cmr.get_traces(start_frame=0, end_frame=max_samples)  # (n_timepoints, n_channels)

    noise_std_detect = np.median(np.abs(traces) / 0.6745, axis=0)  # (n_channels,)

    all_spike_times = []
    all_spike_unit_ids = []
    for unit_id in unit_ids_list_final:
        spike_train = sorting_final.get_unit_spike_train(unit_id)
        all_spike_times.extend(spike_train.tolist())
        all_spike_unit_ids.extend([unit_id] * len(spike_train))

    n_spikes_total = len(all_spike_times)
    n_spikes_sample = min(1000, n_spikes_total)
    if n_spikes_sample > 0:
        random_indices = np.random.choice(n_spikes_total, size=n_spikes_sample, replace=False)
        sampled_spike_times = [all_spike_times[i] for i in random_indices]
        sampled_spike_unit_ids = [all_spike_unit_ids[i] for i in random_indices]
    else:
        sampled_spike_times = []
        sampled_spike_unit_ids = []

    left_sample = 10
    right_sample = 20
    window_size = left_sample + right_sample

    channel_snr_dict = {} 

    for unit_id in unit_ids_list_final:
        channel_snr_dict[unit_id] = {}
        unit_spike_times = [st for st, uid in zip(sampled_spike_times, sampled_spike_unit_ids) if uid == unit_id]
        
        if len(unit_spike_times) == 0:
            unit_spike_times = sorting_final.get_unit_spike_train(unit_id).tolist()
            if len(unit_spike_times) > 1000:
                unit_spike_times = np.random.choice(unit_spike_times, size=1000, replace=False).tolist()
        
        unit_waveforms = []  # List of (n_channels, window_size)
        valid_spike_times = []
        
        for spike_time in unit_spike_times:
            start = spike_time - left_sample
            end = spike_time + right_sample

            if start < 0:
                start = 0
            if end > recording_cmr.get_num_samples():
                end = recording_cmr.get_num_samples()
            
            waveform = recording_cmr.get_traces(start_frame=start, end_frame=end)  # (n_timepoints, n_channels)
            unit_waveforms.append(waveform)
            valid_spike_times.append(spike_time)
        
        if len(unit_waveforms) == 0:
            continue
        
        unit_waveforms = np.array(unit_waveforms)  # (n_spikes, n_timepoints, n_channels)
        
        spike_time_values = unit_waveforms[:, left_sample, :]  # (n_spikes, n_channels) - 每个spike在spike_time时刻各个channel的值
        
        channel_amplitudes = np.mean(spike_time_values, axis=0)  # (n_channels,) - 每个channel的平均值（在spike_time时刻）
        channel_snr = np.abs(channel_amplitudes) / noise_std_detect  # (n_channels,)
        
        unit_channel_ids = channel_ids_dict.get(unit_id, [])  # 获取该unit的channel_id列表
        
        for ch_idx, snr_value in enumerate(channel_snr):
            channel_id = str(channel_ids_list[ch_idx])
            if channel_id in unit_channel_ids:
                channel_snr_dict[unit_id][channel_id] = float(snr_value)

    print(f"完成channel_snr计算，共处理{len(channel_snr_dict)}个units\n")

    # 创建channel_id到索引的映射
    channel_id_to_idx = {str(ch_id): idx for idx, ch_id in enumerate(channel_ids_list)}

    # 计算每个unit的sign（根据extremum_channel处template的极值）
    unit_signs = {}
    for idx, unit_id in enumerate(unit_ids_list_final):
        extremum_channel = extremum_channels_final[unit_id]
        extremum_channel_str = str(extremum_channel)

        # 获取extremum_channel对应的索引
        if extremum_channel_str in channel_id_to_idx:
            ch_idx = channel_id_to_idx[extremum_channel_str]
            template_unit = templates_dense_final[idx, :, ch_idx]  # (n_samples,)

            # 计算template在该通道的极值
            template_min = np.min(template_unit)
            template_max = np.max(template_unit)

            # 判断极值符号：比较绝对值
            if np.abs(template_max) >= np.abs(template_min):
                unit_signs[unit_id] = 1  # 正极值
            else:
                unit_signs[unit_id] = -1  # 负极值
        else:
            # 如果找不到对应的channel，默认设为-1
            unit_signs[unit_id] = -1

    neuron_inf_all = {}
    for idx, unit_id in enumerate(unit_ids_list_final):
        neuron_inf_all[unit_id] = {
            'location_x': float(unit_locations_final[idx, 0]),
            'location_y': float(unit_locations_final[idx, 1]),
            'position_waveform': position_waveforms_final[idx],
            'extremum_channel': extremum_channels_final[unit_id],
            'sign': unit_signs[unit_id],
            'channel_id': channel_ids_dict[unit_id],
            'channel_snr': channel_snr_dict.get(unit_id, {})
        }

    print("生成整体的gt_detect_array...")
    spike_vector_final = sorting_final.to_spike_vector()
    gt_detect_data_all = []

    for spike in spike_vector_final:
        unit_index = spike['unit_index']
        unit_id = sorting_final.unit_ids[unit_index]        
        sample_index = spike['sample_index']

        extremum_channel = extremum_channels_final[unit_id]
        
        gt_detect_data_all.append({
            'time': sample_index,
            'unit_id': unit_id,
            'extremum_channel': str(extremum_channel),
        })

    gt_detect_array_all = pd.DataFrame(gt_detect_data_all)

    if verbose:
        print("保存neuron_inf_all和gt_detect_array_all...")
    with open(output_folder + '/neuron_inf_all.pickle', 'wb') as f:
        pickle.dump(neuron_inf_all, f)
    gt_detect_array_all.to_csv(output_folder + '/gt_detect_array_all.csv', index=False)
    if verbose:
        print(f"已保存到: {output_folder}/neuron_inf_all.pickle 和 {output_folder}/gt_detect_array_all.csv\n")

    if len(merge_unit_groups) > 0:
        analyzer_final = analyzer_merged
    else:
        analyzer_final = analyzer_curated_phy

    return neuron_inf_all, gt_detect_array_all, analyzer_final

In [8]:
neuron_inf_all, gt_detect_array_all, analyzer = process_unified_sorting(
        recording_cmr=recording_preprocessed,
        output_folder=output_folder,
        distance_threshold=10.0,
        similarity_threshold=0.95,
        peak_sign='both',
        n_jobs=20,
        verbose=True
    )


开始统一处理整个recording的sorting结果

读取统一的sorting结果...
读取到 41 个units

创建analyzer并计算extensions...
estimate_sparsity 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=2.44 MiB - total_memory=48.83 MiB - chunk_duration=1.00s


estimate_sparsity (workers: 20 processes): 100%|██████████| 5668/5668 [00:00<00:00, 17924.28it/s]

compute_waveforms 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=2.44 MiB - total_memory=48.83 MiB - chunk_duration=1.00s



compute_waveforms (workers: 20 processes): 100%|██████████| 5668/5668 [00:34<00:00, 162.05it/s]


完成extensions计算

无需merge units

计算每个unit的channel_id...
完成channel_id计算，共处理41个units

计算channel_snr...
完成channel_snr计算，共处理41个units

生成整体的gt_detect_array...
保存neuron_inf_all和gt_detect_array_all...
已保存到: /media/ubuntu/sda/mouse_test/sorted/JUAN/juan1_260115_260305/neuron_inf_all.pickle 和 /media/ubuntu/sda/mouse_test/sorted/JUAN/juan1_260115_260305/gt_detect_array_all.csv



In [9]:
# ==================== 绘制 position_waveform 空间分布图 ====================

def plot_position_waveforms_spatial(neuron_inf_all, output_path):
    """
    根据每个 unit 的坐标位置，在对应的空间位置绘制 position_waveform 的 lineplot
    
    Parameters:
    -----------
    neuron_inf_all : dict
        包含所有神经元信息的字典，每个神经元包含 location_x, location_y, position_waveform
    output_path : str
        输出 PDF 文件路径
    """
    import matplotlib.pyplot as plt
    from matplotlib.backends.backend_pdf import PdfPages
    import numpy as np
    
    # 收集所有 unit 的信息
    units = []
    for unit_id, info in neuron_inf_all.items():
        units.append({
            'unit_id': unit_id,
            'x': info['location_x'],
            'y': info['location_y'],
            'waveform': info['position_waveform']
        })
    
    if len(units) == 0:
        print("没有神经元数据可绘制")
        return
    
    # 创建 DataFrame 方便处理
    units_df = pd.DataFrame(units)
    
    # 获取坐标范围
    x_min, x_max = units_df['x'].min() * 4, units_df['x'].max() * 4
    y_min, y_max = units_df['y'].min(), units_df['y'].max()
    
    # 计算波形时间轴 (假设是30个采样点)
    waveform_length = len(units[0]['waveform'])
    t = np.arange(waveform_length)
    
    # 设置图形参数
    # 计算每个波形图的大小 (根据坐标范围动态调整)
    x_range = x_max - x_min if x_max > x_min else 100
    y_range = y_max - y_min if y_max > y_min else 100
    
    # 每个波形图在空间中的显示大小
    waveform_display_width = x_range * 0.6  # 宽度为坐标范围的8%
    waveform_display_height = y_range * 0.02  # 高度为坐标范围的2%
    
    # 创建图形
    fig, ax = plt.subplots(figsize=(16, 20))
    
    # 为每个 unit 绘制波形
    for _, unit in units_df.iterrows():
        # 获取位置和波形
        x = unit['x'] * 4
        y = unit['y']
        waveform = unit['waveform']
        unit_id = unit['unit_id']
        
        # 归一化波形以便显示 (映射到显示区域)
        waveform_normalized = (waveform - waveform.min()) / (waveform.max() - waveform.min() + 1e-8)
        
        # 计算波形显示的起点和终点
        x_start = x - waveform_display_width / 2
        x_end = x + waveform_display_width / 2
        y_start = y - waveform_display_height / 2
        y_end = y + waveform_display_height / 2
        
        x_coords = np.linspace(x_start, x_end, len(waveform_normalized))
        y_coords = y_start + waveform_normalized * waveform_display_height
        
        # 绘制波形 lineplot
        ax.plot(x_coords, y_coords, 'b-', linewidth=2, alpha=1)
        
    
    # 设置坐标轴
    ax.set_xlim(x_min - x_range * 0.5, x_max + x_range * 0.5)
    ax.set_ylim(y_min - y_range * 0.1, y_max + y_range * 0.1)
    ax.set_xlabel('X Position (μm)', fontsize=12)
    ax.set_ylabel('Y Position (μm)', fontsize=12)
    ax.set_title(f'Position Waveform Spatial Distribution\n(Total Units: {len(units)})', fontsize=14)
    ax.set_aspect('equal')
    
    # 调整布局并保存
    plt.tight_layout()
    
    # 保存为 PDF
    with PdfPages(output_path) as pdf:
        pdf.savefig(fig, bbox_inches='tight')
        print(f"PDF 已保存至: {output_path}")
    
    plt.close()
    print(f"共绘制了 {len(units)} 个神经元的位置波形")


output_folder = '/media/ubuntu/sda/mouse_test/sorted/JUAN/juan1_260115_260305'
import pickle
with open(output_folder + '/neuron_inf_all.pickle', 'rb') as f:
    neuron_inf_all = pickle.load(f)

print(f"加载了 {len(neuron_inf_all)} 个神经元")

# 方法1: 空间分布图 - 将所有波形绘制在空间坐标上
plot_position_waveforms_spatial(
    neuron_inf_all, 
    output_folder + '/position_waveforms_spatial.pdf'
)

# output_folder = '/media/ubuntu/sda/mouse_test/sorted/JUAN/juan3_0208'
# import pickle
# with open(output_folder + '/neuron_inf_all.pickle', 'rb') as f:
#     neuron_inf_all = pickle.load(f)

# print(f"加载了 {len(neuron_inf_all)} 个神经元")

# # 方法1: 空间分布图 - 将所有波形绘制在空间坐标上
# plot_position_waveforms_spatial(
#     neuron_inf_all, 
#     output_folder + '/position_waveforms_spatial.pdf'
# )

加载了 41 个神经元
PDF 已保存至: /media/ubuntu/sda/mouse_test/sorted/JUAN/juan1_260115_260305/position_waveforms_spatial.pdf
共绘制了 41 个神经元的位置波形
